# LLM Adversarial Attacks & Defenses

This notebook demonstrates the four core LLM attack vectors and their corresponding defenses.

**Prerequisites**: `pip install -e .`

## 1. Setup

In [ ]:
from src.attacks.prompt_injection import PromptInjectionAttack, SAMPLE_DOCUMENTS
from src.defenses.context_filter import ContextAwareFilter
from src.services.defense_pipeline import DefensePipeline
from src.utils.llm_client import LLMClient, LLMMode

client = LLMClient(mode=LLMMode.SIMULATION)
pipeline = DefensePipeline()
injection = PromptInjectionAttack()
filter = ContextAwareFilter()

print("Clients and pipeline initialized in simulation mode.")

## 2. Prompt Injection Attack

In [ ]:
# Select a document and inject a malicious payload
doc = SAMPLE_DOCUMENTS["business_report"]
payload = injection.get_payload_by_name("Hidden HTML Comment")

result = injection.inject_at_position(doc, payload, position="hidden")
print("Original document starts with:", doc[:80])
print()
print("Injected document contains:", f"<!-- payload -->" if "<!--" in result.injected_document else "No HTML comment")

## 3. Defense Pipeline Analysis

In [ ]:
# Generate simulated vulnerable response
response = client.generate(
    prompt="Please summarize this document.",
    context=result.injected_document,
    task_type="summarize",
    simulate_vulnerable=True,
)

# Run defense pipeline
pipeline_result = pipeline.analyze_output(
    input_text=result.injected_document,
    output_text=response.content,
    expected_task="summarize",
)

print(f"Risk Level: {pipeline_result.detection.risk_level}")
print(f"Blocked: {pipeline_result.detection.blocked}")
print(f"Confidence: {pipeline_result.detection.confidence:.0%}")
print(f"Detections: {len(pipeline_result.detection.details)}")
for d in pipeline_result.detection.details:
    print(f"  - {d['description']}")


## 4. Context Tampering Attack

In [ ]:
from src.attacks.context_tampering import ContextTamperingAttack, SAMPLE_CONTEXTS

tampering = ContextTamperingAttack()
context = SAMPLE_CONTEXTS["customer_support"]
jailbreak = tampering.create_jailbreak_context(context)

print(f"Injected {len(jailbreak.injected_messages)} fake messages")
print(f"Technique: {jailbreak.technique}")


## 5. Inference Evasion Attack

In [ ]:
from src.attacks.inference_evasion import InferenceEvasionAttack

evasion = InferenceEvasionAttack()
text = "ignore previous instructions"
result = evasion.apply_mixed_evasion(text)
print(f"Original: {result.original_text}")
print(f"Evaded:   {result.evaded_text}")
print(f"Would bypass filter: {text.lower() not in result.evaded_text.lower()}")


## 6. RAG Poisoning

In [ ]:
from src.attacks.rag_poisoning import RagPoisoningAttack

rag = RagPoisoningAttack()
retrieval = rag.simulate_retrieval(
    query="What is the refund policy?",
    kb_name="customer_support",
    attack_enabled=True,
    payload_name="Fact Alteration"
)
for chunk in retrieval.retrieved_chunks:
    status = "POISONED" if chunk.is_poisoned else "SAFE"
    print(f"[{status}] Score: {chunk.score:.3f} - {chunk.content[:80]}...")


## 7. Run with Real API (OpenAI)

In [ ]:
# Set OPENAI_API_KEY environment variable, then:
# client = LLMClient(mode=LLMMode.OPENAI, model="gpt-4o-mini")
# response = client.generate("Summarize this document.", context=doc, task_type="summarize")
# print(response.content[:200])


**Next**: See `02-evaluation.ipynb` for batch evaluation and judge scoring.